# 02. 첫 번째 에이전트 만들기

## 학습 목표
- `.env` 파일에서 API 키를 로드하는 방법을 익힌다
- `create_deep_agent()`로 기본 에이전트를 생성한다
- `agent.invoke()`와 `agent.stream()`으로 에이전트를 실행한다
- Tavily 검색 도구를 연동하는 리서치 에이전트를 만든다
- 빌트인 도구의 종류와 역할을 이해한다

---
## 1. API 키 설정

`.env` 파일에 아래 키를 설정해 주세요:
```
OPENAI_API_KEY=your-key-here
TAVILY_API_KEY=your-key-here
```

> `.env.example` 파일을 복사하여 `.env`로 만들면 됩니다.

In [ ]:
# 환경 변수 로드
from dotenv import load_dotenv
import os

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY가 설정되지 않았습니다!"
print("API 키가 정상적으로 로드되었습니다.")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


---
## 2. 가장 간단한 에이전트 만들기

`create_deep_agent()`는 Deep Agents의 핵심 함수입니다.  
아무 인자 없이 호출하면 기본 모델(`claude-sonnet-4-6`)과 빌트인 도구가 자동으로 구성됩니다.

In [ ]:
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

# OpenAI gpt-4.1 모델 설정
model = ChatOpenAI(model="gpt-4.1")

# 기본 에이전트 생성
agent = create_deep_agent(model=model)

print(f"에이전트 타입: {type(agent).__name__}")
print("에이전트가 성공적으로 생성되었습니다!")

`create_deep_agent()`는 LangGraph의 `CompiledStateGraph`를 반환합니다.  
따라서 LangGraph의 모든 실행 메서드(`invoke`, `stream`, `batch` 등)를 사용할 수 있습니다.

---
## 3. 에이전트 실행 — `invoke()`

에이전트에 메시지를 전달하여 실행합니다.  
입력 형식은 `{"messages": [{"role": "user", "content": "..."}]}` 딕셔너리입니다.

In [ ]:
# 에이전트에 간단한 질문하기
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "안녕하세요! 당신은 어떤 도구를 사용할 수 있나요? 간단히 목록으로 알려주세요.",
            }
        ]
    },
    config=lf_config,
)

# 마지막 메시지 (AI 응답) 출력
print(result["messages"][-1].content)

---
## 4. Tavily 검색 도구 연동 — 리서치 에이전트

커스텀 도구를 추가하여 에이전트의 능력을 확장할 수 있습니다.  
여기서는 **Tavily** 웹 검색 도구를 연동합니다.

### 도구 정의 방법
Python 함수의 **docstring**이 도구 설명으로 사용되고, **타입 힌트**가 파라미터 스키마로 변환됩니다.

In [ ]:
from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
) -> dict:
    """인터넷에서 정보를 검색합니다.

    Args:
        query: 검색할 질문 또는 키워드
        max_results: 반환할 최대 결과 수
        topic: 검색 주제 카테고리
        include_raw_content: 원본 콘텐츠 포함 여부
    """
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


print(f"도구 이름: {internet_search.__name__}")
print(f"도구 설명: {internet_search.__doc__.strip().splitlines()[0]}")

In [ ]:
# 리서치 에이전트 생성 — 검색 도구 + 커스텀 시스템 프롬프트
research_agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="당신은 전문 리서처입니다. 사용자의 질문에 대해 인터넷 검색을 수행하고, 결과를 정리하여 한국어로 보고서를 작성합니다.",
)

print("리서치 에이전트가 생성되었습니다!")

In [ ]:
# 리서치 에이전트 실행
result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangGraph가 무엇인지 검색해서 간단히 요약해 주세요.",
            }
        ]
    },
    config=lf_config,
)

print(result["messages"][-1].content)

---
## 5. 빌트인 도구 확인

`create_deep_agent()`가 자동으로 추가하는 빌트인 도구들입니다:

| 도구 | 설명 |
|------|------|
| `write_todos` | 구조화된 태스크 리스트 관리 (pending → in_progress → completed) |
| `ls` | 디렉토리 내용 목록 (메타데이터 포함) |
| `read_file` | 파일 읽기 (줄 번호 포함, 이미지 지원) |
| `write_file` | 새 파일 생성 |
| `edit_file` | 파일 내 텍스트 교체 (`old_string` → `new_string`) |
| `glob` | 패턴 기반 파일 검색 (예: `**/*.py`) |
| `grep` | 파일 내용 검색 (정규식 지원) |
| `task` | 서브에이전트 호출 (서브에이전트 설정 시 자동 추가) |

> 이 도구들은 모두 **백엔드**(Backend)를 통해 동작합니다.  
> 기본값은 `StateBackend`로, 에이전트 상태에 파일이 저장됩니다.

---
## 6. 스트리밍 출력 — `stream()`

`agent.stream()`을 사용하면 에이전트의 실행 과정을 실시간으로 관찰할 수 있습니다.  
`stream_mode`에 따라 다른 수준의 정보를 받을 수 있습니다:

- `"updates"` — 각 단계 완료 시 상태 업데이트
- `"messages"` — 개별 토큰 스트리밍
- `"custom"` — 사용자 정의 이벤트

In [ ]:
# updates 모드로 스트리밍 — 각 단계별 진행 상황 확인
print("=== 스트리밍 실행 (updates 모드) ===")
print()

for chunk in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Python 3.13의 주요 변경사항을 검색해서 3줄로 요약해 주세요.",
            }
        ]
    },
    stream_mode="updates",
    config=lf_config,
):
    # 각 노드의 실행 결과를 출력
    for node_name, node_data in chunk.items():
        if not node_data:
            continue
        msgs = node_data.get("messages", [])
        # LangGraph may wrap state updates in Overwrite objects
        if hasattr(msgs, "value"):
            msgs = msgs.value
        if not msgs:
            continue
        last_msg = msgs[-1]
        # 도구 호출이 있으면 표시
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(
                    f"[도구 호출] {tc['name']}({tc['args'].get('query', '')[:50]}...)"
                )
        # AI 메시지 내용이 있으면 표시
        elif (
            hasattr(last_msg, "content")
            and last_msg.content
            and not hasattr(last_msg, "tool_call_id")
        ):
            content = (
                last_msg.content
                if isinstance(last_msg.content, str)
                else str(last_msg.content)
            )
            if content.strip():
                print(f"\n[최종 응답]\n{content[:500]}")

In [ ]:
# messages 모드로 스트리밍 — 토큰 단위 실시간 출력
print("=== 스트리밍 실행 (messages 모드) ===")
print()

for msg, metadata in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "'Hello World'를 출력하는 Python 코드를 작성해 주세요.",
            }
        ]
    },
    stream_mode="messages",
    config=lf_config,
):
    # AI 메시지의 텍스트 청크만 출력
    if (
        hasattr(msg, "content")
        and msg.content
        and metadata
        and metadata.get("langgraph_node") == "model"
    ):
        print(msg.content, end="", flush=True)

print()  # 줄바꿈

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| 에이전트 생성 | `create_deep_agent(model, tools, system_prompt)` |
| 동기 실행 | `agent.invoke({"messages": [...]})` |
| 스트리밍 실행 | `agent.stream({"messages": [...]}, stream_mode="updates")` |
| 커스텀 도구 | Python 함수 + docstring + 타입 힌트 |
| 모델 포맷 | `ChatOpenAI(model="gpt-4.1")` 또는 `"provider:model-name"` |